# Dimension Modeling
**Purpose:** Create star schema dimensions for Direct Lake semantic model 

# dim_date - Calendar Dimension

In [14]:
%%sql
CREATE OR REPLACE TABLE dim_date_raw AS
SELECT DISTINCT service_date AS calendar_date
FROM gold_zone_day_metrics
UNION
SELECT DISTINCT pickup_date AS calendar_date
FROM gold_zone_hour_metrics;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 17, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [15]:
%%sql
CREATE OR REPLACE TABLE dim_date (
  date_key INT,
  calendar_date DATE,
  day_of_week INT,
  day_name STRING,
  is_weekend BOOLEAN,
  week_of_year INT,
  month INT,
  month_name STRING,
  quarter INT,
  year INT
) USING DELTA;

INSERT OVERWRITE dim_date
SELECT 
  CAST(CONCAT(YEAR(calendar_date), 
              LPAD(MONTH(calendar_date), 2, '0'), 
              LPAD(DAY(calendar_date), 2, '0')) AS INT) AS date_key,
  calendar_date,
  DAYOFWEEK(calendar_date) AS day_of_week,
  date_format(calendar_date, 'EEEE') AS day_name,       -- "Monday", "Tuesday"
  (DAYOFWEEK(calendar_date) IN (1, 7)) AS is_weekend,
  weekofyear(calendar_date) AS week_of_year,            
  MONTH(calendar_date) AS month,
  date_format(calendar_date, 'MMMM') AS month_name,      -- "January", "February"
  quarter(calendar_date) AS quarter,                    
  YEAR(calendar_date) AS year
FROM dim_date_raw
ORDER BY calendar_date;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 19, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [16]:
%%sql
SELECT 
  COUNT(*) as row_count,
  MIN(calendar_date) as date_min, 
  MAX(calendar_date) as date_max,
  MIN(date_key) as min_date_key,
  MAX(date_key) as max_date_key,
  SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) as weekend_days,
  MIN(day_name) as first_day_name,
  MIN(month_name) as first_month_name
FROM dim_date;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 20, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 8 fields>

# dim_time - Hour Dimension

In [19]:
%%sql
-- dim_time: Hour attributes (24 rows, trivial)
CREATE OR REPLACE TABLE dim_time (
  time_key INT,
  pickup_hour INT,
  hour_label STRING,
  is_peak_hour BOOLEAN,
  part_of_day STRING
) USING DELTA;

INSERT OVERWRITE dim_time
SELECT DISTINCT
  pickup_hour AS time_key,
  pickup_hour,
  CONCAT(LPAD(pickup_hour, 2, '0'), ':00-', LPAD(pickup_hour, 2, '0'), ':59') AS hour_label,
  (pickup_hour BETWEEN 7 AND 10 OR pickup_hour BETWEEN 17 AND 20) AS is_peak_hour,
  CASE 
    WHEN pickup_hour < 6 THEN 'Night'
    WHEN pickup_hour < 12 THEN 'Morning'
    WHEN pickup_hour < 17 THEN 'Afternoon'
    ELSE 'Evening'
  END AS part_of_day
FROM gold_zone_hour_metrics
WHERE pickup_hour IS NOT NULL
ORDER BY pickup_hour;


StatementMeta(, 3a1a1be6-a822-4e39-b9ee-6e61cfda999d, 30, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
%%sql
SELECT 
  COUNT(*) as row_count,
  MIN(pickup_hour) as min_hour,
  MAX(pickup_hour) as max_hour,
  SUM(CASE WHEN is_peak_hour THEN 1 ELSE 0 END) as peak_hours,
  COLLECT_LIST(part_of_day) as parts_of_day
FROM dim_time;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 13, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 5 fields>

# dim_zone - NYC Taxi Zone Dimension

In [23]:
%%sql
-- dim_zone: NYC Taxi zones with activity metrics
CREATE OR REPLACE TABLE dim_zone (
  zone_id INT,
  zone_description STRING,
  total_trips_hourly BIGINT,
  total_trips_daily BIGINT
) USING DELTA;

INSERT OVERWRITE dim_zone
WITH hourly_stats AS (
  SELECT 
    zone_id,
    SUM(trips_count) AS total_trips_hourly
  FROM gold_zone_hour_metrics
  GROUP BY zone_id
),
daily_stats AS (
  SELECT 
    zone_id,
    SUM(trips) AS total_trips_daily
  FROM gold_zone_day_metrics
  GROUP BY zone_id
)
SELECT 
  COALESCE(h.zone_id, d.zone_id) AS zone_id,
  CONCAT('Zone_', CAST(COALESCE(h.zone_id, d.zone_id) AS STRING)) AS zone_description,
  COALESCE(h.total_trips_hourly, 0) AS total_trips_hourly,
  COALESCE(d.total_trips_daily, 0) AS total_trips_daily
FROM hourly_stats h
FULL OUTER JOIN daily_stats d ON h.zone_id = d.zone_id
WHERE (COALESCE(h.total_trips_hourly, 0) + COALESCE(d.total_trips_daily, 0)) > 10  -- Active zones only
ORDER BY (COALESCE(h.total_trips_hourly, 0) + COALESCE(d.total_trips_daily, 0)) DESC;


StatementMeta(, 3a1a1be6-a822-4e39-b9ee-6e61cfda999d, 35, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
%%sql
SELECT 
  COUNT(*) as zone_count,
  MIN(zone_id) as min_zone,
  MAX(zone_id) as max_zone,
  SUM(total_trips_hourly) as total_hourly_trips,
  SUM(total_trips_daily) as total_daily_trips
FROM dim_zone;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 14, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 5 fields>

In [7]:
select count(*),is_rain from gold_zone_day_metrics
group by is_rain


StatementMeta(, f461e96b-bb18-4e1b-871d-61638c4020d4, 8, Finished, Available, Finished)

<Spark SQL result set with 2 rows and 2 fields>

In [17]:
%%sql
select count(*) from silver_zone_hour_metrics_streaming;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 21, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

In [32]:
%%sql
CREATE TABLE IF NOT EXISTS gold_zone_hour_metrics_streaming;


StatementMeta(, 73e07d55-63e8-4db3-9c01-7a3dbe4c7e67, 35, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [25]:
%%sql
INSERT INTO gold_zone_hour_metrics_streaming
SELECT
  CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date)      AS pickup_date,
  HOUR(to_timestamp(tpep_pickup_datetime))                                 AS pickup_hour,
  CAST(PULocationID AS int)                                                AS zone_id,
  COUNT(*)                                                                 AS trips_count,
  SUM(total_amount)                                                        AS total_revenue,
  AVG(total_amount)                                                        AS avg_total_amount,
  AVG(trip_distance)                                                       AS avg_trip_distance,
  AVG(
      (UNIX_TIMESTAMP(to_timestamp(tpep_dropoff_datetime)) -
       UNIX_TIMESTAMP(to_timestamp(tpep_pickup_datetime))) / 60.0
  )                                                                        AS avg_trip_duration_min,
  SUM(tip_amount)                                                          AS tip_amount_total,
  CASE WHEN SUM(total_amount) = 0 THEN 0.0
       ELSE SUM(tip_amount) / SUM(total_amount)
  END                                                                      AS tip_share_pct,
  CASE
    WHEN dayofweek(date_trunc('day', to_timestamp(tpep_pickup_datetime))) IN (1,7)
    THEN TRUE ELSE FALSE
  END                                                                      AS is_weekend,
  CURRENT_TIMESTAMP                                                        AS load_timestamp
FROM silver_zone_hour_metrics_streaming
WHERE
  PULocationID IS NOT NULLa
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime IS NOT NULL
  AND total_amount > 0
GROUP BY
  CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date),
  HOUR(to_timestamp(tpep_pickup_datetime)),
  CAST(PULocationID AS int);
-- using the same query for scheduled runs.

StatementMeta(, 73e07d55-63e8-4db3-9c01-7a3dbe4c7e67, 26, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
%%sql
select count(*) from gold_zone_hour_metrics_streaming
-- where pickup_ts is not null limit 10;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 26, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

In [30]:
%%sql

SELECT 
  ROUND(avg_total_amount, 2) AS avg_total_amount_2dp,
  ROUND(avg_trip_distance, 2) AS avg_trip_distance_2dp,
  ROUND(avg_trip_duration_min, 2) AS avg_trip_duration_min_2dp,
  ROUND(tip_share_pct, 4) AS tip_share_pct_4dp
FROM gold_zone_hour_metrics_streaming
LIMIT 10;


StatementMeta(, 73e07d55-63e8-4db3-9c01-7a3dbe4c7e67, 33, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 4 fields>

# Creating Log table

In [35]:
%%sql
CREATE TABLE IF NOT EXISTS pipeline_run_log (
  run_id          string,
  run_time        timestamp,
  source_table    string,
  target_table    string,
  rows_written    bigint,
  status          string,
  message         string
);


StatementMeta(, 73e07d55-63e8-4db3-9c01-7a3dbe4c7e67, 38, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql
SELECT * FROM pipeline_run_log ORDER BY run_time DESC;


StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 5, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 7 fields>

# Simple DQ metrics table

In [37]:
%%sql
CREATE TABLE IF NOT EXISTS dq_streaming_checks (
  check_time        timestamp,
  window_start      timestamp,
  window_end        timestamp,
  total_trips       bigint,
  avg_total_amount  double,
  bad_distance_pct  double
);


StatementMeta(, 73e07d55-63e8-4db3-9c01-7a3dbe4c7e67, 40, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql
select * from dq_streaming_checks

StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 7, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 6 fields>

In [3]:
from pyspark.sql.functions import col, max as max_, to_timestamp, expr


StatementMeta(, 0b5fcdcc-f403-41f4-b364-2a1f633a8193, 5, Finished, Available, Finished)

In [5]:
%%sql
ALTER TABLE gold_zone_hour_metrics_streaming
ADD COLUMN pickup_ts timestamp;


StatementMeta(, 0b5fcdcc-f403-41f4-b364-2a1f633a8193, 7, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql
INSERT INTO gold_zone_hour_metrics_streaming
SELECT
  CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date)      AS pickup_date,
  HOUR(to_timestamp(tpep_pickup_datetime))                                 AS pickup_hour,
  CAST(PULocationID AS int)                                                AS zone_id,
  COUNT(*)                                                                 AS trips_count,
  SUM(total_amount)                                                        AS total_revenue,
  AVG(total_amount)                                                        AS avg_total_amount,
  AVG(trip_distance)                                                       AS avg_trip_distance,
  AVG(
      (UNIX_TIMESTAMP(to_timestamp(tpep_dropoff_datetime)) -
       UNIX_TIMESTAMP(to_timestamp(tpep_pickup_datetime))) / 60.0
  )                                                                        AS avg_trip_duration_min,
  SUM(tip_amount)                                                          AS tip_amount_total,
  CASE WHEN SUM(total_amount) = 0 THEN 0.0
       ELSE SUM(tip_amount) / SUM(total_amount)
  END                                                                      AS tip_share_pct,
  CASE
    WHEN dayofweek(date_trunc('day', to_timestamp(tpep_pickup_datetime))) IN (1,7)
    THEN TRUE ELSE FALSE
  END                                                                      AS is_weekend,
  CURRENT_TIMESTAMP                                                        AS load_timestamp,
  date_trunc(
      'hour',
      to_timestamp(tpep_pickup_datetime)
  )                                                                        AS pickup_ts
FROM silver_zone_hour_metrics_streaming
WHERE
  PULocationID IS NOT NULL
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime IS NOT NULL
  AND total_amount > 0
GROUP BY
  CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date),
  HOUR(to_timestamp(tpep_pickup_datetime)),
  CAST(PULocationID AS int),
  date_trunc('hour', to_timestamp(tpep_pickup_datetime));


StatementMeta(, 0b5fcdcc-f403-41f4-b364-2a1f633a8193, 8, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
from pyspark.sql.functions import col, to_timestamp, max as max_

# 1) Read streaming gold
gold_df = spark.table("gold_zone_hour_metrics_streaming")

if gold_df.head(1):
    last_processed_ts = gold_df.select(max_("pickup_ts").alias("max_ts")).collect()[0]["max_ts"]
else:
    last_processed_ts = None

print("Last processed pickup_ts:", last_processed_ts)

# 2) Read streaming silver and add pickup_ts
silver_df = spark.table("silver_zone_hour_metrics_streaming") \
    .withColumn("pickup_ts", to_timestamp("tpep_pickup_datetime"))

if last_processed_ts is not None:
    incr_df = silver_df.filter(col("pickup_ts") > last_processed_ts)
else:
    incr_df = silver_df

incr_df.createOrReplaceTempView("new_streaming_silver")


StatementMeta(, 0b5fcdcc-f403-41f4-b364-2a1f633a8193, 11, Finished, Available, Finished)

Last processed pickup_ts: 2022-06-07 03:00:00


In [9]:
%%sql
describe dim_date

StatementMeta(, 716082b7-d990-48db-8d8f-b4d0f9151635, 11, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 3 fields>